# 🫁 Módulo 2: Biofísica del Astronauta, Climatología MEDA y Capas Multi-Misión
### Marswalk 3D — NASA Space Apps Challenge

---

## 1. Bioenergética en Traje Espacial ($PLSS$) y Gravedad Marciana
El costo de desplazamiento de un astronauta en Marte está dominado por la masa del traje presurizado ($\sim 140 	ext{ kg}$ total) y la gravedad reducida ($g = 3.71 	ext{ m/s}^2$).

Adaptamos el modelo de **Pandolf, Givoni y Goldman (1977)** y **Margaria (1963)** para locomoción humana en trajes de presión:

$$\dot{V}_{O_2} = \dot{V}_{	ext{base}} 	imes f_{	ext{pendiente}}(	heta) 	imes C_{	ext{climb}}\left(rac{\Delta z}{d}ight) 	imes M_{	ext{run}} 	imes P_{	ext{térmico}}$$

Donde:
- $\dot{V}_{	ext{base}} = 50 	ext{ L/h}$ ($pprox 0.0139 	ext{ L/s}$) tasa metabólica basal en marcha nominal ($1.2 	ext{ m/s}$).
- $C_{	ext{climb}}$ es el trabajo gravitacional contra el desnivel.
- $f_{	ext{pendiente}}(	heta)$ penaliza fuertemente laderas $> 15^\circ$ y prohíbe el paso por encima del umbral crítico de estabilidad de $	heta_{	ext{safe}} = 25^\circ$.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dem_pipeline import DEMPipeline
from src.physiology import PhysiologicalModel
from src.multimission import MultiMissionLayerProcessor
from src.solar_physics import MartianSolarPhysics

data_dir = repo_root / "data" / "jezero_test"
dem_pipe = DEMPipeline(data_dir)
physio = PhysiologicalModel(data_dir / "config_fisio.json")

print(f"Reserva de O2 en mochila PLSS: {physio.o2_reserve_l} Litros")
print(f"Tasa basal de consumo: {physio.o2_base_consumption_l_per_h} L/h ({physio.base_rate_l_s*1000:.2f} mL/s)")
print(f"Velocidad de marcha nominal: {physio.walk_speed_ms} m/s | Carrera: {physio.run_speed_ms} m/s")
print(f"Umbral de pendiente máxima segura: {physio.max_safe_slope_deg}°")


## 2. Curvas de Consumo Metabólico vs. Inclinación del Terreno


In [ ]:
slopes = np.linspace(0, 30, 100)
dist_step = 100.0  # 100 metros de tramo

costs_walk_up = []
costs_walk_down = []
costs_run_up = []

for s in slopes:
    dz = dist_step * np.sin(np.radians(s))
    o2_up, _, _ = physio.calculate_segment_cost(dist_step, dz, s, running=False)
    o2_down, _, _ = physio.calculate_segment_cost(dist_step, -dz, s, running=False)
    o2_run, _, _ = physio.calculate_segment_cost(dist_step, dz, s, running=True)
    
    costs_walk_up.append(o2_up)
    costs_walk_down.append(o2_down)
    costs_run_up.append(o2_run)

plt.figure(figsize=(10, 5))
plt.plot(slopes, costs_walk_up, color="#55f29b", lw=2, label="Marcha Ascendente (1.2 m/s)")
plt.plot(slopes, costs_walk_down, color="#66fcf1", lw=2, ls="-.", label="Marcha Descendente (1.2 m/s)")
plt.plot(slopes, costs_run_up, color="#f6c945", lw=2, ls="--", label="Carrera Ascendente (2.5 m/s)")
plt.axvline(25.0, color="#ff3855", lw=1.5, ls=":", label="Límite Biomecánico Crítico (25°)")

plt.title("Gasto de Oxígeno (L) por cada 100 metros recorridos en Marte", fontweight="bold")
plt.xlabel("Pendiente del Terreno (°)")
plt.ylabel("Consumo de Oxígeno (L)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 3. Fusión Multi-Misión: Mineralogía CRISM e Inercia THEMIS

Un explorador humano planifica su ruta para investigar la **astrobiología del cráter**:
1. **CRISM (MRO):** Detección de filosilicatos (arcillas hidratadas Fe/Mg) en el lecho del paleolago y carbonatos marginales en las terrazas de evaporación.
2. **THEMIS (Mars Odyssey):** Diferenciación entre roca madre consolidada (alta inercia térmica, excelente tracción) y dunas de arena eólica suelta (baja inercia térmica, trampa de regolito).


In [ ]:
processor = MultiMissionLayerProcessor(dem_pipe)
layer_mats = processor.generate_layer_matrices(rows=160, cols=160)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

im1 = axes[0].imshow(layer_mats["crism_clays"], cmap="GnBu", extent=[-dem_pipe.half_w/1000, dem_pipe.half_w/1000, -dem_pipe.half_d/1000, dem_pipe.half_d/1000])
axes[0].set_title("CRISM: Arcillas / Filosilicatos", fontweight="bold")
axes[0].set_xlabel("Distancia X (km)")
axes[0].set_ylabel("Distancia Z (km)")
fig.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(layer_mats["themis_thermal_inertia"], cmap="YlOrBr", extent=[-dem_pipe.half_w/1000, dem_pipe.half_w/1000, -dem_pipe.half_d/1000, dem_pipe.half_d/1000])
axes[1].set_title("THEMIS: Inercia Térmica (Roca vs Arena)", fontweight="bold")
axes[1].set_xlabel("Distancia X (km)")
fig.colorbar(im2, ax=axes[1])

im3 = axes[2].imshow(layer_mats["risk_index"], cmap="Reds", extent=[-dem_pipe.half_w/1000, dem_pipe.half_w/1000, -dem_pipe.half_d/1000, dem_pipe.half_d/1000], vmin=0, vmax=1.2)
axes[2].set_title("Mapa de Riesgo Geotécnico", fontweight="bold")
axes[2].set_xlabel("Distancia X (km)")
fig.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()


## 4. Efemérides Solares y Atenuación Atmosférica MEDA ($	au$)

La distancia Marte-Sol $r(L_s)$ varía debido a la excentricidad orbital ($e=0.0934$). La irradiancia en superficie se rige por la **Ley de Beer-Lambert** con la opacidad de polvo atmosférico $	au$ medida por la estación meteorológica MEDA de Perseverance:

$$I_{	ext{superficie}} = rac{1361 	ext{ W/m}^2}{r(L_s)^2} \cdot \sin(lpha_\odot) \cdot e^{-	au \cdot m(lpha_\odot)}$$


In [ ]:
solar = MartianSolarPhysics(lat_deg=18.38, lon_deg=77.58)

# Simular ciclo diurno (LTST de 06:00 a 18:00) bajo condiciones nominales (tau=0.45) y tormenta (tau=3.8)
hours = np.linspace(6, 18, 50)
irr_nominal = [solar.compute_solar_state(ls_deg=142.0, ltst_h=h, dust_tau=0.45)["surface_irradiance_wm2"] for h in hours]
irr_storm   = [solar.compute_solar_state(ls_deg=142.0, ltst_h=h, dust_tau=3.80)["surface_irradiance_wm2"] for h in hours]

plt.figure(figsize=(10, 4.5))
plt.plot(hours, irr_nominal, color="#f6c945", lw=2.5, label="Nominal (τ = 0.45)")
plt.plot(hours, irr_storm, color="#ff3855", lw=2.5, ls="--", label="Tormenta de Polvo MEDA (τ = 3.80)")
plt.title("Irradiancia Solar Superficial en Cráter Jezero (Ls = 142°)", fontweight="bold")
plt.xlabel("Hora Solar Local Verdadera (LTST)")
plt.ylabel("Flujo Radiativo (W/m²)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
